In [5]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive

# 1. Mount Drive & Load Data
drive.mount('/content/drive/')
data = pd.read_csv('/content/drive/MyDrive/SlashMark_Tasks/Restaurant_Reviews.tsv', delimiter='\t', quoting=3)

# 2. Text Preprocessing (Tokenize, Stop-words, Lemmatize)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve the LookupError

lemmatizer = WordNetLemmatizer()
corpus = []
for i in range(len(data)):
    review = re.sub('[^a-zA-Z]', ' ', data['Review'][i])
    review = review.lower()
    review = nltk.word_tokenize(review) # Tokenize
    review = [lemmatizer.lemmatize(word) for word in review if word not in set(stopwords.words('english'))] # Lemmatize
    corpus.append(' '.join(review))

# 3. Feature Extraction (TF-IDF)
tfidf = TfidfVectorizer(max_features=1500)
X = tfidf.fit_transform(corpus).toarray()
y = data.iloc[:, 1].values

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

# 5. Classical ML (Logistic Regression)
classifier = LogisticRegression(random_state=0)
classifier.fit(X_train, y_train)

# 6. Evaluation (Accuracy and F1)
y_pred = classifier.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 7. Basic Model Deployment Inference
def predict_sentiment(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    text_words = [lemmatizer.lemmatize(w) for w in nltk.word_tokenize(text) if w not in set(stopwords.words('english'))]
    processed_text = ' '.join(text_words)
    prediction = classifier.predict(tfidf.transform([processed_text]).toarray())
    return "Positive" if prediction[0] == 1 else "Negative"

print("\nDeployment Test:", predict_sentiment("The food was amazing and the service was great!"))

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Accuracy: 75.00%
F1 Score: 72.83%

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.86      0.77        97
           1       0.83      0.65      0.73       103

    accuracy                           0.75       200
   macro avg       0.76      0.75      0.75       200
weighted avg       0.76      0.75      0.75       200


Deployment Test: Positive
